# AI PARSE DOCUMENT + AI PREP SEARCH

Requisites
-- ai_parse_document is available in DBR 17.1 or serverless runtime
-- ai_prep_search has to be enabled in previews of the workspace

## Understand what `ai_parse_document` does

`ai_parse_document` reads binary file content (e.g., PDFs) and returns a **nested STRUCT** with the following key fields:

| Field | Type | Description |
|---|---|---|
| `document.elements` | `ARRAY<STRUCT>` | Array of parsed elements — each element has a `content` (text), `type` (paragraph, image, table, etc.), and positional metadata |
| `document.pages` | `ARRAY<STRUCT>` | Per-page metadata including page number and dimensions |

### Key takeaway
The parsed output contains **nested arrays and structs** that need to be exploded/transformed before use:
- Use `transform()` + `array_join()` to flatten element text into a single string
- Use `explode()` to create one row per element or per page
- Use `::ARRAY<STRUCT<content: STRING>>` casting to access typed fields


# Building a knowledge base from PDF documentation using Databricks Vector search (RAG)

For this solution we will need to leverage on three functions 'ai_parse_document', ai_prep_search and 'ai_query' 

Thanksfully, all the information required is available to us as PDF. These pdf are stored in our volume. (/manuales)

We'll parse them and save them in our Vector Search, to improve our agent capabilities



## 1.1/ Create our chunck view

Let's first create our table.

In [0]:
# Configuration - catalog & schema are passed as job/DAB parameters (with defaults for interactive runs)
dbutils.widgets.text("catalog", "stable_classic_6kvrb7_catalog", "Catalog")
dbutils.widgets.text("schema", "cesar_cordoba", "Schema")
dbutils.widgets.text("volume", "manuales", "Volume")

catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")
volume = dbutils.widgets.get("volume")
# Workspace URL derived dynamically so this works in any workspace (no hardcoded host)
workspace_url = spark.conf.get("spark.databricks.workspaceUrl")

print(f"Using catalog={catalog}, schema={schema}, workspace_url={workspace_url}")

In [0]:
# The object may already exist as a VIEW from a previous run; CREATE OR REPLACE TABLE
# cannot replace a view, so drop whatever is there first (view or table).
for _obj in ("VIEW", "TABLE"):
    try:
        spark.sql(f"DROP {_obj} IF EXISTS {catalog}.{schema}.{volume}_chunked_document_view")
    except Exception:
        pass

spark.sql(f"""
CREATE OR REPLACE TABLE {catalog}.{schema}.{volume}_chunked_document_view AS
WITH parsed_documents AS (
  SELECT ai_parse_document(
    content,
    map(
      'imageOutputPath', '/Volumes/{catalog}/{schema}/{volume}/images',
      'descriptionElementTypes', '*',
      'pageRange', '1-40'
    )
  ) AS parsed,
  path as filename,
  modificationTime
  FROM READ_FILES('/Volumes/{catalog}/{schema}/{volume}/*.pdf', format => 'binaryFile')
),
prepped_documents AS (
  SELECT 
  filename,
  modificationTime,
  ai_prep_search(parsed) AS result 
  FROM parsed_documents
)
SELECT
  filename,
  modificationTime,
  chunk.value:chunk_id::STRING AS chunk_id,
  chunk.value:chunk_to_embed::STRING AS chunk_to_embed,
  chunk.value:pages AS pages
FROM
  prepped_documents,
  LATERAL variant_explode(prepped_documents.result:document.contents) AS chunk
""")

## Working with images with multimodal llms


### Define a prompt

In [0]:
prompt = """Analyze the provided document image.
1. Determine if this page contains any diagrams, figures, charts, flowcharts, illustrations, schematics, or any visual/graphical element (anything that is NOT purely text or tables of text).
2. If the page contains one or more visual elements, set "has_visual_content" to true and describe each one in the "visuals" array: what type it is (diagram, flowchart, chart, figure, illustration, etc.) and a detailed description of its content — what it depicts, labels, relationships, data shown, etc.
3. If the page is purely text, tables of text, or otherwise has NO visual/graphical elements, set "has_visual_content" to false and leave the "visuals" array empty.
4. DO NOT invent visuals that are not there.
Return the results adhering to the JSON schema."""

simplified_schema = {
    "type": "json_schema",
    "json_schema": {
        "name": "document_visual_extraction",
        "strict": "true",
        "schema": {
            "type": "object",
            "properties": {
                "has_visual_content": {"type": "boolean"},
                "visuals": {
                    "type": "array",
                    "items": {
                        "type": "object",
                        "properties": {
                            "visual_type": {"type": "string"},
                            "description": {"type": "string"}
                        },
                        "required": ["visual_type", "description"]
                    }
                }
            },
            "required": ["has_visual_content", "visuals"]
        }
    }
}

### Test the AI over the images

We recommend playing around with the models. Claude Sonnet 4-5 is quite good cost/performance. Nevertheless, for some files, you might find other models more suitable/ more robust

## 1.2/ Generate image metadata

In [0]:
from pyspark.sql.functions import col, element_at, split, expr, lit
import json

# 1. Read images and extract the parent folder (which is the PDF name)
image_df = (
    spark.read.format("binaryFile")
    .load(f"/Volumes/{catalog}/{schema}/{volume}/images/")
    .withColumn("id", element_at(split(col("path"), "/"), -1))
)

# 2. Run the Gemini query with the simplified schema (one row per page)
extraction_df = image_df.withColumn("prompt", lit(prompt)).withColumn(
    "page_results",
    expr(f"""
    ai_query(
        'databricks-claude-sonnet-4-5',
        prompt,
        files => content,
        responseFormat => '{json.dumps(simplified_schema)}'
    )
"""),
)

from pyspark.sql.functions import col, from_json, explode_outer
from pyspark.sql.types import StructType, StructField, StringType, BooleanType, ArrayType

# Spark schema matching the simplified_schema from defined prompt
spark_schema = StructType([
    StructField("has_visual_content", BooleanType()),
    StructField("visuals", ArrayType(StructType([
        StructField("visual_type", StringType()),
        StructField("description", StringType())
    ])))
])

# Parse the JSON string and explode visuals into individual rows
parsed_df = (
    extraction_df
    .withColumn("parsed", from_json(col("page_results"), spark_schema))
    .select(
        "id", "path",
        col("parsed.has_visual_content").alias("has_visual_content"),
        explode_outer(col("parsed.visuals")).alias("visual")
    )
    .select(
        "id", "path", "has_visual_content",
        col("visual.visual_type").alias("visual_type"),
        col("visual.description").alias("description")
    )
    .filter('has_visual_content==True')
)

# Materialize the image analysis to a table so ai_query runs exactly once and the images
# (already written by the parse step above) are read from a stable, static location.
(
    parsed_df.write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{catalog}.{schema}.{volume}_image_visuals")
)
parsed_df = spark.table(f"{catalog}.{schema}.{volume}_image_visuals")
parsed_df.display()

## 1.3/ Join Chunk view + image data

In [0]:
from pyspark.sql.functions import col, explode_outer, regexp_replace, concat, lit

chunks_df = spark.table(f"{catalog}.{schema}.{volume}_chunked_document_view")

# Explode pages array to get one row per page per chunk, keep filename & modificationTime
chunks_exploded = (
    chunks_df
    .withColumn("doc_uri", 
        concat(lit(f"https://{workspace_url}/ajax-api/2.0/fs/files"), regexp_replace(col("filename"), "^dbfs:", ""))
    )
    .select("filename","modificationTime", "doc_uri", "chunk_id", "chunk_to_embed",
            explode_outer(col("pages").cast('array<struct<image_uri:string, page_id:string>>')).alias("page"))
    .select(
        "filename", "modificationTime", "doc_uri","chunk_id", "chunk_to_embed", 
        col("page.image_uri").alias("image_uri"),
        col("page.page_id").alias("page_id")
    )
)

# Strip dbfs: prefix from parsed_df.path to match image_uri
parsed_clean = parsed_df.withColumn("image_uri", regexp_replace(col("path"), "^dbfs:", ""))

# Left join: enrich chunks with visual descriptions where available
aggregated_info_df = (
    chunks_exploded
    .join(parsed_clean, on="image_uri", how="left")
    .select(
        "filename", "modificationTime", "doc_uri", "chunk_id", "chunk_to_embed", 
        "image_uri", "page_id", "has_visual_content", "visual_type", "description"
    )
)

aggregated_info_df.display()

In [0]:
# Ensure a clean target table. We avoid a GENERATED ALWAYS AS IDENTITY column so we can
# supply our own primary key ("id") below, which the Vector Search delta-sync index needs.
spark.sql(f"DROP TABLE IF EXISTS {catalog}.{schema}.{volume}_document_with_images_parsed")

In [0]:
from pyspark.sql.functions import monotonically_increasing_id

# Add a stable, non-null unique key required by the Vector Search delta-sync index
(
    aggregated_info_df
        .withColumn("id", monotonically_increasing_id()) 
        .write.format("delta") 
        .option("delta.enableChangeDataFeed", "true") 
        .option("overwriteSchema", "true") 
        .mode("overwrite") 
        .saveAsTable(f"{catalog}.{schema}.{volume}_document_with_images_parsed")
)